In [1]:
#Connect to database

import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/cohort.db")
print("✓ Connected to cohort.db")

# Confirm tables
print(pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
).to_string(index=False))

✓ Connected to cohort.db
       name
subscribers
   activity


In [2]:
#Overall business KPIs

# ── Query 1: Business Summary ────────────────────────────────
query = """
SELECT
    COUNT(DISTINCT s.subscriber_id)         AS total_subscribers,
    COUNT(DISTINCT s.plan)                  AS total_plans,
    COUNT(DISTINCT s.cohort_month)          AS total_cohorts,
    ROUND(SUM(s.monthly_price), 2)          AS total_mrr_at_signup,
    ROUND(AVG(s.monthly_price), 2)          AS avg_monthly_price
FROM subscribers s
"""
pd.read_sql(query, conn)

,total_subscribers,total_plans,total_cohorts,total_mrr_at_signup,avg_monthly_price
0,2000,4,23,177830.0,88.92


In [3]:
# Subscribers per cohort month

# ── Query 2: Cohort Size by Month ────────────────────────────
# How many new subscribers signed up each month?

query = """
SELECT
    cohort_month,
    COUNT(*)                                AS new_subscribers,
    ROUND(AVG(monthly_price), 2)            AS avg_plan_price,
    ROUND(SUM(monthly_price), 2)            AS cohort_mrr
FROM subscribers
GROUP BY cohort_month
ORDER BY cohort_month
"""
pd.read_sql(query, conn)

,cohort_month,new_subscribers,avg_plan_price,cohort_mrr
0,2022-02,8,62.75,502.0
1,2022-03,17,108.41,1843.0
2,2022-04,31,76.42,2369.0
3,2022-05,37,88.73,3283.0
4,2022-06,33,87.48,2887.0
5,2022-07,54,146.59,7916.0
6,2022-08,59,82.90,4891.0
7,2022-09,71,97.45,6919.0
8,2022-10,65,79.62,5175.0
9,2022-11,70,91.00,6370.0


In [4]:
#Retention by cohort

# ── Query 3: Cohort Retention Table ──────────────────────────
# The most important query in this project
# Shows what % of each cohort is still active at each month

query = """
WITH cohort_size AS (
    SELECT cohort_month,
           COUNT(DISTINCT subscriber_id)    AS cohort_subscribers
    FROM subscribers
    GROUP BY cohort_month
),
monthly_active AS (
    SELECT cohort_month,
           month_number,
           COUNT(DISTINCT subscriber_id)    AS active_subscribers
    FROM activity
    GROUP BY cohort_month, month_number
)
SELECT
    m.cohort_month,
    m.month_number,
    m.active_subscribers,
    c.cohort_subscribers,
    ROUND(m.active_subscribers * 100.0 /
          c.cohort_subscribers, 1)          AS retention_pct
FROM monthly_active m
JOIN cohort_size c ON m.cohort_month = c.cohort_month
ORDER BY m.cohort_month, m.month_number
"""
pd.read_sql(query, conn)

,cohort_month,month_number,active_subscribers,cohort_subscribers,retention_pct
0,2022-02,0,8,8,100.0
1,2022-02,1,5,8,62.5
2,2022-02,2,4,8,50.0
3,2022-02,3,4,8,50.0
4,2022-02,4,4,8,50.0
...,...,...,...,...,...
547,2023-12,8,112,170,65.9
548,2023-12,9,105,170,61.8
549,2023-12,10,101,170,59.4
550,2023-12,11,96,170,56.5


In [5]:
#Retention by plan

# ── Query 4: Retention by Plan ───────────────────────────────
# Do higher tier plans retain better?

query = """
WITH plan_cohort_size AS (
    SELECT plan,
           COUNT(DISTINCT subscriber_id)    AS total_subscribers
    FROM subscribers
    GROUP BY plan
),
plan_monthly_active AS (
    SELECT plan,
           month_number,
           COUNT(DISTINCT subscriber_id)    AS active_subscribers
    FROM activity
    GROUP BY plan, month_number
)
SELECT
    p.plan,
    p.month_number,
    p.active_subscribers,
    c.total_subscribers,
    ROUND(p.active_subscribers * 100.0 /
          c.total_subscribers, 1)           AS retention_pct
FROM plan_monthly_active p
JOIN plan_cohort_size c ON p.plan = c.plan
WHERE p.month_number IN (0, 1, 3, 6, 12)
ORDER BY p.plan, p.month_number
"""
pd.read_sql(query, conn)

,plan,month_number,active_subscribers,total_subscribers,retention_pct
0,Basic,0,814,814,100.0
1,Basic,1,718,814,88.2
2,Basic,3,574,814,70.5
3,Basic,6,436,814,53.6
4,Basic,12,281,814,34.5
5,Business,0,371,371,100.0
6,Business,1,359,371,96.8
7,Business,3,340,371,91.6
8,Business,6,313,371,84.4
9,Business,12,257,371,69.3


In [6]:
#Monthly recurring revenue (MRR)

# ── Query 5: MRR by Month ────────────────────────────────────
# Total revenue from active subscribers each month

query = """
SELECT
    activity_month,
    COUNT(DISTINCT subscriber_id)           AS active_subscribers,
    ROUND(SUM(monthly_price), 2)            AS mrr,
    ROUND(AVG(monthly_price), 2)            AS avg_revenue_per_user
FROM activity
GROUP BY activity_month
ORDER BY activity_month
"""
pd.read_sql(query, conn)

,activity_month,active_subscribers,mrr,avg_revenue_per_user
0,2022-02,8,502.0,62.75
1,2022-03,22,2208.0,100.36
2,2022-04,51,4469.0,87.63
3,2022-05,85,7565.0,89.00
4,2022-06,112,10078.0,89.98
5,2022-07,156,17484.0,112.08
6,2022-08,203,21067.0,103.78
7,2022-09,256,26724.0,104.39
8,2022-10,303,30667.0,101.21
9,2022-11,350,34650.0,99.00


In [7]:
#Churn analysis by plan

# ── Query 6: Churn Analysis by Plan ─────────────────────────
# How many subscribers churned from each plan?

query = """
WITH last_active AS (
    SELECT subscriber_id,
           MAX(month_number)               AS last_month
    FROM activity
    GROUP BY subscriber_id
),
churn_flag AS (
    SELECT s.subscriber_id,
           s.plan,
           s.monthly_price,
           s.cohort_month,
           l.last_month,
           CASE WHEN l.last_month < 24
                THEN 1 ELSE 0
           END                             AS churned
    FROM subscribers s
    JOIN last_active l
      ON s.subscriber_id = l.subscriber_id
)
SELECT
    plan,
    COUNT(*)                               AS total_subscribers,
    SUM(churned)                           AS churned,
    COUNT(*) - SUM(churned)                AS retained,
    ROUND(SUM(churned) * 100.0 /
          COUNT(*), 1)                     AS churn_rate_pct,
    ROUND(AVG(last_month), 1)              AS avg_months_active
FROM churn_flag
GROUP BY plan
ORDER BY churn_rate_pct DESC
"""
pd.read_sql(query, conn)

,plan,total_subscribers,churned,retained,churn_rate_pct,avg_months_active
0,Basic,814,789,25,96.9,8.3
1,Pro,707,660,47,93.4,11.3
2,Business,371,325,46,87.6,14.2
3,Enterprise,108,92,16,85.2,15.3


In [8]:
#Revenue retention

# ── Query 7: Revenue Retention by Cohort ─────────────────────
# What % of each cohort's original MRR is still active?
# More important than subscriber retention for SaaS businesses

query = """
WITH cohort_mrr AS (
    SELECT s.cohort_month,
           ROUND(SUM(s.monthly_price), 2)  AS original_mrr
    FROM subscribers s
    GROUP BY s.cohort_month
),
monthly_mrr AS (
    SELECT a.cohort_month,
           a.month_number,
           ROUND(SUM(a.monthly_price), 2)  AS active_mrr
    FROM activity a
    GROUP BY a.cohort_month, a.month_number
)
SELECT
    m.cohort_month,
    m.month_number,
    m.active_mrr,
    c.original_mrr,
    ROUND(m.active_mrr * 100.0 /
          c.original_mrr, 1)               AS revenue_retention_pct
FROM monthly_mrr m
JOIN cohort_mrr c ON m.cohort_month = c.cohort_month
WHERE m.month_number IN (0, 1, 3, 6, 12)
ORDER BY m.cohort_month, m.month_number
"""
pd.read_sql(query, conn)

,cohort_month,month_number,active_mrr,original_mrr,revenue_retention_pct
0,2022-02,0,502.0,502.0,100.0
1,2022-02,1,365.0,502.0,72.7
2,2022-02,3,336.0,502.0,66.9
3,2022-02,6,307.0,502.0,61.2
4,2022-02,12,307.0,502.0,61.2
...,...,...,...,...,...
110,2023-12,0,13820.0,13820.0,100.0
111,2023-12,1,13164.0,13820.0,95.3
112,2023-12,3,12372.0,13820.0,89.5
113,2023-12,6,11568.0,13820.0,83.7


In [9]:
#Acquisition channel performance

# ── Query 8: Channel Performance ─────────────────────────────
# Which acquisition channel produces the best retaining subscribers?

query = """
WITH channel_size AS (
    SELECT channel,
           COUNT(DISTINCT subscriber_id)   AS total_subscribers,
           ROUND(AVG(monthly_price), 2)    AS avg_plan_price
    FROM subscribers
    GROUP BY channel
),
channel_active AS (
    SELECT s.channel,
           COUNT(DISTINCT CASE WHEN a.month_number >= 6
                          THEN a.subscriber_id END) AS active_at_month6
    FROM subscribers s
    JOIN activity a ON s.subscriber_id = a.subscriber_id
    GROUP BY s.channel
)
SELECT
    c.channel,
    c.total_subscribers,
    c.avg_plan_price,
    ca.active_at_month6,
    ROUND(ca.active_at_month6 * 100.0 /
          c.total_subscribers, 1)          AS month6_retention_pct
FROM channel_size c
JOIN channel_active ca ON c.channel = ca.channel
ORDER BY month6_retention_pct DESC
"""
pd.read_sql(query, conn)

,channel,total_subscribers,avg_plan_price,active_at_month6,month6_retention_pct
0,Social Media,412,92.40,302,73.3
1,Referral,377,94.09,252,66.8
2,Organic Search,410,82.85,269,65.6
3,Paid Ads,405,90.41,264,65.2
4,Direct,396,85.11,255,64.4


In [10]:
#Window function: subscriber lifetime value

# ── Query 9: Subscriber LTV with Window Functions ────────────
# Calculates cumulative revenue per subscriber over time
# Shows window function skills — SUM OVER ORDER BY

query = """
WITH subscriber_monthly AS (
    SELECT subscriber_id,
           plan,
           monthly_price,
           month_number,
           SUM(monthly_price) OVER (
               PARTITION BY subscriber_id
               ORDER BY month_number
           )                              AS cumulative_revenue
    FROM activity
)
SELECT
    plan,
    month_number,
    ROUND(AVG(cumulative_revenue), 2)    AS avg_cumulative_ltv,
    COUNT(DISTINCT subscriber_id)        AS active_subscribers
FROM subscriber_monthly
WHERE month_number IN (0, 1, 3, 6, 12)
GROUP BY plan, month_number
ORDER BY plan, month_number
"""
pd.read_sql(query, conn)

,plan,month_number,avg_cumulative_ltv,active_subscribers
0,Basic,0,29.0,814
1,Basic,1,58.0,718
2,Basic,3,116.0,574
3,Basic,6,203.0,436
4,Basic,12,377.0,281
5,Business,0,149.0,371
6,Business,1,298.0,359
7,Business,3,596.0,340
8,Business,6,1043.0,313
9,Business,12,1937.0,257


In [11]:
#Save SQL file and close

sql_queries = """
-- ============================================================
-- SAAS COHORT RETENTION ANALYSIS — SQL Queries
-- Database: data/cohort.db
-- Tables: subscribers, activity
-- ============================================================

-- 1. BUSINESS SUMMARY
SELECT
    COUNT(DISTINCT subscriber_id)       AS total_subscribers,
    COUNT(DISTINCT cohort_month)        AS total_cohorts,
    ROUND(SUM(monthly_price), 2)        AS total_mrr_at_signup
FROM subscribers;

-- 2. COHORT RETENTION TABLE
WITH cohort_size AS (
    SELECT cohort_month,
           COUNT(DISTINCT subscriber_id) AS cohort_subscribers
    FROM subscribers
    GROUP BY cohort_month
),
monthly_active AS (
    SELECT cohort_month,
           month_number,
           COUNT(DISTINCT subscriber_id) AS active_subscribers
    FROM activity
    GROUP BY cohort_month, month_number
)
SELECT
    m.cohort_month,
    m.month_number,
    ROUND(m.active_subscribers * 100.0 /
          c.cohort_subscribers, 1)      AS retention_pct
FROM monthly_active m
JOIN cohort_size c ON m.cohort_month = c.cohort_month
ORDER BY m.cohort_month, m.month_number;

-- 3. CHURN BY PLAN
WITH last_active AS (
    SELECT subscriber_id,
           MAX(month_number)            AS last_month
    FROM activity
    GROUP BY subscriber_id
)
SELECT
    s.plan,
    COUNT(*)                            AS total_subscribers,
    ROUND(SUM(CASE WHEN l.last_month < 24
              THEN 1 ELSE 0 END) * 100.0 /
          COUNT(*), 1)                  AS churn_rate_pct
FROM subscribers s
JOIN last_active l ON s.subscriber_id = l.subscriber_id
GROUP BY s.plan
ORDER BY churn_rate_pct DESC;

-- 4. SUBSCRIBER LTV WITH WINDOW FUNCTION
WITH subscriber_monthly AS (
    SELECT subscriber_id, plan, monthly_price, month_number,
           SUM(monthly_price) OVER (
               PARTITION BY subscriber_id
               ORDER BY month_number
           )                            AS cumulative_revenue
    FROM activity
)
SELECT
    plan,
    month_number,
    ROUND(AVG(cumulative_revenue), 2)   AS avg_cumulative_ltv
FROM subscriber_monthly
WHERE month_number IN (0, 1, 3, 6, 12)
GROUP BY plan, month_number
ORDER BY plan, month_number;
"""

with open("../sql/cohort_queries.sql", "w") as f:
    f.write(sql_queries)

conn.close()
print("✓ SQL file saved to sql/cohort_queries.sql")
print("✓ Connection closed")

✓ SQL file saved to sql/cohort_queries.sql
✓ Connection closed
